<a href="https://www.kaggle.com/code/avikdas567/the-2026-neurogolf-championship?scriptVersionId=312246199" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Imports

In [1]:
import os
import json
import numpy as np
from tqdm import tqdm
import zipfile

import onnx
from onnx import helper, TensorProto

## Paths

In [2]:
BASE_PATH = "/kaggle/input/competitions/neurogolf-2026"
TASK_PATH = BASE_PATH

TASK_FILES = sorted([f for f in os.listdir(TASK_PATH) if f.endswith(".json")])

print("Total tasks:", len(TASK_FILES))

Total tasks: 400


## Utility Functions

In [3]:
def load_task(path):
    with open(path, "r") as f:
        return json.load(f)

def grid_to_onehot(grid):
    h, w = len(grid), len(grid[0])
    tensor = np.zeros((10, 30, 30), dtype=np.float32)
    for i in range(h):
        for j in range(w):
            tensor[grid[i][j], i, j] = 1
    return tensor

## Transformation Detection

In [4]:
def detect_transformation(task):
    train = task["train"]

    def to_np(x):
        return np.array(x)

    # Identity
    if all(pair["input"] == pair["output"] for pair in train):
        return ("identity", None)

    # Color mapping
    mapping = {}
    valid = True
    for pair in train:
        inp = to_np(pair["input"])
        out = to_np(pair["output"])
        if inp.shape != out.shape:
            valid = False
            break
        for i in range(inp.shape[0]):
            for j in range(inp.shape[1]):
                if inp[i,j] not in mapping:
                    mapping[inp[i,j]] = out[i,j]
                elif mapping[inp[i,j]] != out[i,j]:
                    valid = False
                    break
    if valid:
        return ("color_map", mapping)

    # Rotations
    for k in [1,2,3]:
        if all(np.array_equal(np.rot90(to_np(p["input"]), k=k), to_np(p["output"])) for p in train):
            return ("rot", k)

    # Flips
    if all(np.array_equal(np.fliplr(to_np(p["input"])), to_np(p["output"])) for p in train):
        return ("flip_lr", None)

    if all(np.array_equal(np.flipud(to_np(p["input"])), to_np(p["output"])) for p in train):
        return ("flip_ud", None)

    # Crop (bounding box)
    def crop(x):
        x = np.array(x)
        ys, xs = np.where(x != 0)
        if len(ys) == 0:
            return x
        return x[min(ys):max(ys)+1, min(xs):max(xs)+1]

    if all(np.array_equal(crop(p["input"]), np.array(p["output"])) for p in train):
        return ("crop", None)

    # Object color isolation (very common ARC pattern)
    def isolate(x):
        x = np.array(x)
        vals, counts = np.unique(x[x != 0], return_counts=True)
        if len(vals) == 0:
            return x
        main = vals[np.argmax(counts)]
        return (x == main).astype(int) * main

    if all(np.array_equal(isolate(p["input"]), np.array(p["output"])) for p in train):
        return ("isolate", None)

    return ("unknown", None)

## ONNX Builders (Tiny Networks)

### Identity (VERY CHEAP)

In [5]:
def build_identity_model(task_id):
    input_tensor = helper.make_tensor_value_info("input", TensorProto.FLOAT, [1,10,30,30])
    output_tensor = helper.make_tensor_value_info("output", TensorProto.FLOAT, [1,10,30,30])

    node = helper.make_node("Identity", ["input"], ["output"])

    graph = helper.make_graph([node], f"task{task_id}", [input_tensor], [output_tensor])
    model = helper.make_model(graph)

    return model

### Color Mapping (1x1 Conv = optimal)

In [6]:
def build_color_map_model(task_id, mapping):
    weight = np.zeros((10,10,1,1), dtype=np.float32)

    for i in range(10):
        if i in mapping:
            weight[mapping[i], i, 0, 0] = 1.0

    W = helper.make_tensor("W", TensorProto.FLOAT, weight.shape, weight.flatten())

    node = helper.make_node(
        "Conv",
        inputs=["input", "W"],
        outputs=["output"],
        kernel_shape=[1,1]
    )

    graph = helper.make_graph(
        [node],
        f"task{task_id}",
        [helper.make_tensor_value_info("input", TensorProto.FLOAT, [1,10,30,30])],
        [helper.make_tensor_value_info("output", TensorProto.FLOAT, [1,10,30,30])],
        initializer=[W]
    )

    return helper.make_model(graph)

## Rotation / Flip (index remap trick)

In [7]:
def build_permutation_model(task_id, mode):
    # NOTE: uses Gather → allowed and cheap
    # We'll precompute index mapping

    # For simplicity, fallback to identity if complex
    return build_identity_model(task_id)

## Solve One Task

In [8]:
def build_identity_model(task_id):
    node = helper.make_node("Identity", ["input"], ["output"])
    graph = helper.make_graph(
        [node],
        f"task{task_id}",
        [helper.make_tensor_value_info("input", TensorProto.FLOAT, [1,10,30,30])],
        [helper.make_tensor_value_info("output", TensorProto.FLOAT, [1,10,30,30])]
    )
    return helper.make_model(graph)


def build_color_map_model(task_id, mapping):
    weight = np.zeros((10,10,1,1), dtype=np.float32)
    for i in range(10):
        if i in mapping:
            weight[mapping[i], i, 0, 0] = 1.0

    W = helper.make_tensor("W", TensorProto.FLOAT, weight.shape, weight.flatten())

    node = helper.make_node("Conv", ["input", "W"], ["output"], kernel_shape=[1,1])

    graph = helper.make_graph(
        [node],
        f"task{task_id}",
        [helper.make_tensor_value_info("input", TensorProto.FLOAT, [1,10,30,30])],
        [helper.make_tensor_value_info("output", TensorProto.FLOAT, [1,10,30,30])],
        initializer=[W]
    )
    return helper.make_model(graph)


def build_mask_model(task_id, color):
    weight = np.zeros((10,10,1,1), dtype=np.float32)
    weight[color, color, 0, 0] = 1.0

    W = helper.make_tensor("W", TensorProto.FLOAT, weight.shape, weight.flatten())

    node = helper.make_node("Conv", ["input", "W"], ["output"], kernel_shape=[1,1])

    graph = helper.make_graph(
        [node],
        f"task{task_id}",
        [helper.make_tensor_value_info("input", TensorProto.FLOAT, [1,10,30,30])],
        [helper.make_tensor_value_info("output", TensorProto.FLOAT, [1,10,30,30])],
        initializer=[W]
    )
    return helper.make_model(graph)


def solve_task(task, task_id):
    t, param = detect_transformation(task)

    # Identity
    if t == "identity":
        return build_identity_model(task_id)

    # Color mapping
    if t == "color_map":
        return build_color_map_model(task_id, param)

    # Crop → fallback identity (safe ONNX)
    if t == "crop":
        return build_identity_model(task_id)

    # Object isolation → build mask
    if t == "isolate":
        # pick dominant color
        vals = []
        for p in task["train"]:
            x = np.array(p["input"])
            v, c = np.unique(x[x != 0], return_counts=True)
            if len(v):
                vals.append(v[np.argmax(c)])
        if len(vals):
            color = max(set(vals), key=vals.count)
            return build_mask_model(task_id, color)

    # Rotations / flips → fallback (safe)
    if t in ["rot", "flip_lr", "flip_ud"]:
        return build_identity_model(task_id)

    # Unknown → safe fallback
    return build_identity_model(task_id)

## Main Loop

In [9]:
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for idx, file in enumerate(tqdm(TASK_FILES)):
    task = load_task(os.path.join(TASK_PATH, file))
    model = solve_task(task, idx+1)

    onnx.save(model, os.path.join(OUTPUT_DIR, f"task{idx+1:03d}.onnx"))

100%|██████████| 400/400 [00:06<00:00, 60.06it/s]


## Zip Submission

In [10]:
zip_path = "/kaggle/working/submission.zip"

with zipfile.ZipFile(zip_path, 'w') as z:
    for f in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, f), f)

print("Submission saved:", zip_path)

Submission saved: /kaggle/working/submission.zip
